In [ ]:
import sys
from pathlib import Path

def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / ".git").exists():
            return p
    raise RuntimeError("Could not locate project root (no .git found above cwd)")

PROJECT_ROOT = _find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))


# Unified Model Evaluation Pipeline

This notebook evaluates all saved model families with one pipeline using the Section 9+ evaluation structure from the incremental-monthly SGD notebook.

Key policies implemented:
- Incremental-scaler families use the model-year scaler when evaluating prior years.
- Standard-scaler families use the evaluation-year scaler.
- Feature preparation is routed by model family to the original notebook-style function.
- Each evaluation table is saved immediately and supports resume (existing table files are skipped and loaded).
- Final-model evaluation supports an optional final scaler when a family saves one separately.

In [ ]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = PROJECT_ROOT
EVAL_DIR = PROJECT_ROOT / 'experiments' / 'evaluation' / 'eval_outputs' / 'unified_eval'
EVAL_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_PATH = ROOT / 'data_split.npz'
if not SPLIT_PATH.exists():
    raise FileNotFoundError(f'Missing split file: {SPLIT_PATH.resolve()}')

split = np.load(SPLIT_PATH)

def _first_existing_key(candidates):
    for key in candidates:
        if key in split.files:
            return key
    return None

train_key = _first_existing_key(['train_pixel_indices', 'train_idx', 'train_indices'])
val_key = _first_existing_key(['val_pixel_indices', 'valid_pixel_indices', 'val_idx', 'val_indices'])
test_key = _first_existing_key(['test_pixel_indices', 'test_idx', 'test_indices'])

if not all([train_key, val_key, test_key]):
    raise KeyError(f'Could not resolve split keys from: {split.files}')

train_pixel_indices = split[train_key]
val_pixel_indices = split[val_key]
test_pixel_indices = split[test_key]

print(f'Loaded splits from {SPLIT_PATH.name}')
print(f'  train: {len(train_pixel_indices):,} | val: {len(val_pixel_indices):,} | test: {len(test_pixel_indices):,}')

# The family registry (which model/scaler/dataset/feature-prep-kind goes with
# which family_id) lives in src/eval/families.py -- moved verbatim, with a
# short_label added to every family. See that module's docstring for why no
# generator replaces the literal, and why short_label subsumes what used to be a
# second, only-partially-covering rename dict in the reporting cell.
from src.eval.families import build_families, open_family_dataset

FAMILIES = build_families(PROJECT_ROOT)

# open_family_dataset (src/eval/families.py) memoizes per dataset_path, which
# matters because 50 families share only 3 zarr files. DATASET_CACHE is the cache
# object it reads and writes; load_family_dataset keeps the notebook's original
# one-argument call shape.
DATASET_CACHE = {}


def load_family_dataset(family_cfg):
    return open_family_dataset(family_cfg, DATASET_CACHE)

print(f'Configured {len(FAMILIES)} model families')

Loaded splits from data_split.npz
  train: 5,597,776 | val: 1,273,437 | test: 1,283,992
Configured 43 model families


In [6]:
# Feature preparation for every model family.
#
# The feature math lives in src/mlp_replay/data.py and the caching/dispatch wrapper
# in src/eval/features.py, so evaluation and training can no longer drift apart.
# This notebook previously carried its own copy of the feature builder, and read the
# baseline family's version by exec-ing it out of SGD Classifier.ipynb at runtime --
# both are gone.
#
# Per-family flags below are unchanged, including the two asymmetries worth knowing:
#   * the baseline family imputes S2 gaps over an expanding window (years <= t),
#     while every other family imputes over all years. That difference predates this
#     refactor; it is preserved here rather than quietly harmonised.
#   * the neighbourhood family sets include_last_year=False.
from src.eval.features import FEATURE_CACHE as _FEATURE_CACHE
from src.eval.features import prepare_features_common

# Cell below reports cache statistics through these names; bind them to the cache's
# own objects so the counters stay live.
FEATURE_CACHE = _FEATURE_CACHE.entries
FEATURE_CACHE_STATS = _FEATURE_CACHE.stats
FEATURE_CACHE_MAX_ENTRIES = _FEATURE_CACHE.max_entries


def prepare_features_prevyears(ds, pixel_indices, year_idx, scaler=None, scaler_mode='auto', family_id=None):
    return prepare_features_common(
        ds=ds,
        pixel_indices=pixel_indices,
        year_idx=year_idx,
        scaler=scaler,
        scaler_mode=scaler_mode,
        include_last_year=True,
        include_monthly=False,
        include_neighbourhood=False,
        family_id=family_id,
    )


def prepare_features_baseline_notebook(ds, pixel_indices, year_idx, scaler=None, scaler_mode='auto', family_id=None):
    """Baseline family: no lag features, no monthly indices, expanding-window imputation.

    Named for the era when this function was read out of SGD Classifier.ipynb by
    exec-ing the notebook JSON. It is a normal call now; the name is kept because
    the family configs reference it.
    """
    _ = family_id
    if scaler_mode not in {'auto', 'fit', 'transform'}:
        raise ValueError(
            "baseline_notebook prep only supports scaler_mode values 'auto', 'fit', and 'transform'."
        )
    if scaler_mode == 'transform' and scaler is None:
        raise ValueError("A fitted scaler is required for baseline_notebook with scaler_mode='transform'.")

    return prepare_features_common(
        ds=ds,
        pixel_indices=pixel_indices,
        year_idx=year_idx,
        scaler=None if scaler_mode == 'fit' else scaler,
        scaler_mode=scaler_mode,
        include_last_year=False,
        include_monthly=False,
        include_neighbourhood=False,
        family_id=None,
        impute_window='expanding',
    )


def prepare_features_monthly(ds, pixel_indices, year_idx, scaler=None, scaler_mode='auto', family_id=None):
    return prepare_features_common(
        ds=ds,
        pixel_indices=pixel_indices,
        year_idx=year_idx,
        scaler=scaler,
        scaler_mode=scaler_mode,
        include_last_year=True,
        include_monthly=True,
        include_neighbourhood=False,
        family_id=family_id,
    )


def prepare_features_neighbourhood(ds, pixel_indices, year_idx, scaler=None, scaler_mode='auto', family_id=None):
    return prepare_features_common(
        ds=ds,
        pixel_indices=pixel_indices,
        year_idx=year_idx,
        scaler=scaler,
        scaler_mode=scaler_mode,
        include_last_year=False,
        include_monthly=False,
        include_neighbourhood=True,
        family_id=family_id,
    )


PREP_FN_BY_KIND = {
    'baseline_notebook': prepare_features_baseline_notebook,
    'prevyears': prepare_features_prevyears,
    'monthly': prepare_features_monthly,
    'neighbourhood': prepare_features_neighbourhood,
}


# Scoring metrics and threshold selection live in src/eval/metrics.py -- moved
# verbatim. best_f1_threshold stays a separate implementation from
# src/thresholds.py on purpose (see that module's docstring); both are preserved
# there now instead of only in this cell.
from src.eval.metrics import best_f1_threshold, compute_metrics

# Table read/write lives in src/eval/tables.py. The JSON twins there carry no
# timestamp, so re-running the pipeline no longer rewrites every tracked JSON with a
# one-line diff -- git status is now a real signal about whether a run had side effects.
from src.eval.tables import load_or_run_table, save_table

print('Feature preparation and helper functions loaded')

Feature preparation and helper functions loaded


In [ ]:
# Model/scaler loading and feature-count validation live in src/eval/artifacts.py --
# pure functions, no free globals, moved verbatim.
from src.eval.artifacts import (
    infer_years_from_models,
    load_final_scaler,
    load_model,
    load_scaler,
    resolve_scaler_year,
    validate_feature_count,
    validate_model_features,
    validate_scaler_features,
)


def compute_threshold_for_pair(model_obj, prep_fn, ds, family_cfg, model_year, eval_year, scaler_obj, rbf_sampler=None, default_threshold=0.5, family_id=None):
    year_to_idx = {int(y): i for i, y in enumerate(ds.year.values)}
    eval_year_idx = year_to_idx[int(eval_year)]
    if scaler_obj is None:
        val_scale_mode = 'none'
    else:
        val_scale_mode = 'transform'

    X_val, y_val, _ = prep_fn(
        ds,
        val_pixel_indices,
        eval_year_idx,
        scaler=scaler_obj,
        scaler_mode=val_scale_mode,
        family_id=family_id,
    )
    if len(X_val) == 0:
        return float(default_threshold), {'source': 'fallback', 'reason': 'empty validation set', 'n_val_samples': 0}

    validate_scaler_features(X_val, scaler_obj, family_cfg['label'], model_year, eval_year, 'validation')

    if rbf_sampler is not None:
        X_val = rbf_sampler.transform(X_val)

    validate_model_features(X_val, model_obj, family_cfg['label'], model_year, eval_year, 'validation')

    y_val_proba = model_obj.predict_proba(X_val)[:, 1]
    return best_f1_threshold(y_val, y_val_proba, default_threshold=default_threshold)


def evaluate_one_pair(model_obj, prep_fn, ds, family_cfg, model_year, eval_year, scaler_obj, threshold, threshold_meta, rbf_sampler=None, family_id=None):
    year_to_idx = {int(y): i for i, y in enumerate(ds.year.values)}
    eval_year_idx = year_to_idx[int(eval_year)]

    if scaler_obj is None:
        scale_mode = 'none'
    else:
        scale_mode = 'transform'

    X_test, y_test, _ = prep_fn(
        ds,
        test_pixel_indices,
        eval_year_idx,
        scaler=scaler_obj,
        scaler_mode=scale_mode,
        family_id=family_id,
    )
    if len(X_test) == 0:
        return None

    validate_scaler_features(X_test, scaler_obj, family_cfg['label'], model_year, eval_year, 'test')

    if rbf_sampler is not None:
        X_test = rbf_sampler.transform(X_test)

    validate_model_features(X_test, model_obj, family_cfg['label'], model_year, eval_year, 'test')

    y_proba = model_obj.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)
    metrics = compute_metrics(y_test, y_pred, y_proba)

    return {
        'model_year': int(model_year),
        'eval_year': int(eval_year),
        'n_test_samples': int(len(y_test)),
        'threshold_used': float(threshold),
        'threshold_source': threshold_meta.get('source', 'unknown'),
        'threshold_reason': threshold_meta.get('reason', ''),
        'n_val_threshold_samples': int(threshold_meta.get('n_val_samples', 0)),
        **metrics,
    }


def evaluate_family(family_id, family_cfg, default_threshold=0.5):
    prep_fn = PREP_FN_BY_KIND[family_cfg['prep_kind']]
    ds = load_family_dataset(family_cfg)
    models_dir = family_cfg['models_dir']

    if not models_dir.exists():
        raise FileNotFoundError(f'Model directory missing: {models_dir.resolve()}')

    years = infer_years_from_models(models_dir, family_cfg['model_template'])
    if not years:
        raise RuntimeError(f'No model years found for {family_id}')

    rbf_sampler = None
    if 'rbf_sampler_path' in family_cfg:
        rbf_path = family_cfg['rbf_sampler_path']
        if rbf_path.exists():
            with open(rbf_path, 'rb') as f:
                rbf_sampler = pickle.load(f)

    family_out_dir = EVAL_DIR / family_id
    family_out_dir.mkdir(parents=True, exist_ok=True)

    status_rows = []

    cache_hits_before = FEATURE_CACHE_STATS['hits']
    cache_misses_before = FEATURE_CACHE_STATS['misses']

    def _run_table_own_year():
        rows = []
        for year in years:
            model_obj = load_model(models_dir, family_cfg['model_template'], year)
            if model_obj is None:
                continue
            scaler_year, scaler_policy = resolve_scaler_year(family_cfg, year, year)
            scaler_obj = load_scaler(models_dir, family_cfg['scaler_template'], scaler_year)

            threshold, threshold_meta = compute_threshold_for_pair(
                model_obj,
                prep_fn,
                ds,
                family_cfg,
                year,
                year,
                scaler_obj,
                rbf_sampler=rbf_sampler,
                default_threshold=default_threshold,
                family_id=family_id,
            )
            result = evaluate_one_pair(
                model_obj,
                prep_fn,
                ds,
                family_cfg,
                year,
                year,
                scaler_obj,
                threshold,
                threshold_meta,
                rbf_sampler=rbf_sampler,
                family_id=family_id,
            )
            if result is None:
                continue
            result.update({
                'family_id': family_id,
                'family_label': family_cfg['label'],
                'table_type': 'own_year',
                'scaler_year_used': int(scaler_year),
                'scaler_policy_used': scaler_policy,
            })
            rows.append(result)
        return pd.DataFrame(rows)

    def _run_table_prior_years():
        rows = []
        for model_year in years:
            model_obj = load_model(models_dir, family_cfg['model_template'], model_year)
            if model_obj is None:
                continue
            for eval_year in [y for y in years if y <= model_year]:
                scaler_year, scaler_policy = resolve_scaler_year(family_cfg, model_year, eval_year)
                scaler_obj = load_scaler(models_dir, family_cfg['scaler_template'], scaler_year)

                threshold, threshold_meta = compute_threshold_for_pair(
                    model_obj,
                    prep_fn,
                    ds,
                    family_cfg,
                    model_year,
                    eval_year,
                    scaler_obj,
                    rbf_sampler=rbf_sampler,
                    default_threshold=default_threshold,
                    family_id=family_id,
                )
                result = evaluate_one_pair(
                    model_obj,
                    prep_fn,
                    ds,
                    family_cfg,
                    model_year,
                    eval_year,
                    scaler_obj,
                    threshold,
                    threshold_meta,
                    rbf_sampler=rbf_sampler,
                    family_id=family_id,
                )
                if result is None:
                    continue
                result.update({
                    'family_id': family_id,
                    'family_label': family_cfg['label'],
                    'table_type': 'prior_years',
                    'scaler_year_used': int(scaler_year),
                    'scaler_policy_used': scaler_policy,
                })
                rows.append(result)
        return pd.DataFrame(rows)

    def _run_table_next_year():
        rows = []
        for model_year in years:
            candidate = model_year + 1
            if candidate not in years:
                continue
            model_obj = load_model(models_dir, family_cfg['model_template'], model_year)
            if model_obj is None:
                continue

            scaler_year, scaler_policy = resolve_scaler_year(family_cfg, model_year, candidate)
            scaler_obj = load_scaler(models_dir, family_cfg['scaler_template'], scaler_year)

            threshold, threshold_meta = compute_threshold_for_pair(
                model_obj,
                prep_fn,
                ds,
                family_cfg,
                model_year,
                candidate,
                scaler_obj,
                rbf_sampler=rbf_sampler,
                default_threshold=default_threshold,
                family_id=family_id,
            )
            result = evaluate_one_pair(
                model_obj,
                prep_fn,
                ds,
                family_cfg,
                model_year,
                candidate,
                scaler_obj,
                threshold,
                threshold_meta,
                rbf_sampler=rbf_sampler,
                family_id=family_id,
            )
            if result is None:
                continue
            result.update({
                'family_id': family_id,
                'family_label': family_cfg['label'],
                'table_type': 'next_year',
                'scaler_year_used': int(scaler_year),
                'scaler_policy_used': scaler_policy,
            })
            rows.append(result)
        return pd.DataFrame(rows)

    def _run_table_cumulative():
        rows = []
        year_to_idx = {int(y): i for i, y in enumerate(ds.year.values)}
        for model_year in years:
            model_obj = load_model(models_dir, family_cfg['model_template'], model_year)
            if model_obj is None:
                continue

            X_blocks = []
            y_blocks = []
            scaler_policies = []
            scaler_years = []
            threshold_rows = []

            for eval_year in [y for y in years if y <= model_year]:
                scaler_year, scaler_policy = resolve_scaler_year(family_cfg, model_year, eval_year)
                scaler_obj = load_scaler(models_dir, family_cfg['scaler_template'], scaler_year)
                if scaler_obj is None and family_cfg['scaler_template'] is not None:
                    continue

                eval_year_idx = year_to_idx[int(eval_year)]
                scale_mode = 'none' if scaler_obj is None else 'transform'
                X_block, y_block, _ = prep_fn(
                    ds,
                    test_pixel_indices,
                    eval_year_idx,
                    scaler=scaler_obj,
                    scaler_mode=scale_mode,
                    family_id=family_id,
                )
                if len(X_block) == 0:
                    continue

                validate_scaler_features(X_block, scaler_obj, family_cfg['label'], model_year, eval_year, 'cumulative_test_block')

                if rbf_sampler is not None:
                    X_block = rbf_sampler.transform(X_block)

                validate_model_features(X_block, model_obj, family_cfg['label'], model_year, eval_year, 'cumulative_test_block')

                X_blocks.append(X_block)
                y_blocks.append(y_block)
                scaler_policies.append(scaler_policy)
                scaler_years.append(int(scaler_year))

                threshold, threshold_meta = compute_threshold_for_pair(
                    model_obj,
                    prep_fn,
                    ds,
                    family_cfg,
                    model_year,
                    eval_year,
                    scaler_obj,
                    rbf_sampler=rbf_sampler,
                    default_threshold=default_threshold,
                    family_id=family_id,
                )
                threshold_rows.append((threshold, threshold_meta))

            if not X_blocks:
                continue

            threshold, threshold_meta = threshold_rows[-1]
            X_test = np.vstack(X_blocks)
            y_test = np.concatenate(y_blocks)

            validate_model_features(X_test, model_obj, family_cfg['label'], model_year, model_year, 'cumulative_test_stack')

            y_proba = model_obj.predict_proba(X_test)[:, 1]
            y_pred = (y_proba >= threshold).astype(int)
            metrics = compute_metrics(y_test, y_pred, y_proba)

            rows.append({
                'family_id': family_id,
                'family_label': family_cfg['label'],
                'table_type': 'cumulative',
                'model_year': int(model_year),
                'eval_year': int(model_year),
                'n_test_samples': int(len(y_test)),
                'years_included': ','.join(str(y) for y in years if y <= model_year),
                'threshold_used': float(threshold),
                'threshold_source': threshold_meta.get('source', 'unknown'),
                'threshold_reason': threshold_meta.get('reason', ''),
                'n_val_threshold_samples': int(threshold_meta.get('n_val_samples', 0)),
                'scaler_year_used': int(scaler_years[-1]),
                'scaler_policy_used': scaler_policies[-1],
                **metrics,
            })
        return pd.DataFrame(rows)

    def _run_table_final_model_each_year():
        final_model = None
        final_model_path = family_cfg.get('final_model_path')
        if final_model_path and final_model_path.exists():
            with open(final_model_path, 'rb') as f:
                final_model = pickle.load(f)
        else:
            return pd.DataFrame([])

        final_scaler = load_final_scaler(family_cfg)
        rows = []
        for eval_year in years:
            if final_scaler is not None:
                scaler_obj = final_scaler
                scaler_year = max(years)
                scaler_policy = 'family_final_scaler'
            else:
                scaler_year, scaler_policy = resolve_scaler_year(family_cfg, max(years), eval_year)
                scaler_obj = load_scaler(models_dir, family_cfg['scaler_template'], scaler_year)
            threshold, threshold_meta = compute_threshold_for_pair(
                final_model,
                prep_fn,
                ds,
                family_cfg,
                max(years),
                eval_year,
                scaler_obj,
                rbf_sampler=rbf_sampler,
                default_threshold=default_threshold,
                family_id=family_id,
            )
            result = evaluate_one_pair(
                final_model,
                prep_fn,
                ds,
                family_cfg,
                max(years),
                eval_year,
                scaler_obj,
                threshold,
                threshold_meta,
                rbf_sampler=rbf_sampler,
                family_id=family_id,
            )
            if result is None:
                continue
            result.update({
                'family_id': family_id,
                'family_label': family_cfg['label'],
                'table_type': 'final_model_each_year',
                'scaler_year_used': int(scaler_year),
                'scaler_policy_used': scaler_policy,
            })
            rows.append(result)
        return pd.DataFrame(rows)

    table_runners = {
        'own_year': _run_table_own_year,
        'prior_years': _run_table_prior_years,
        'next_year': _run_table_next_year,
        'cumulative': _run_table_cumulative,
        'final_model_each_year': _run_table_final_model_each_year,
    }

    table_frames = {}
    for table_name, table_fn in table_runners.items():
        table_path = family_out_dir / f'{family_id}_{table_name}.csv'
        df_table, skipped = load_or_run_table(table_path, table_fn)
        table_frames[table_name] = df_table
        status_rows.append({
            'family_id': family_id,
            'table_name': table_name,
            'skipped_existing': bool(skipped),
            'row_count': int(len(df_table)),
            'csv_path': str(table_path),
        })

    combined = pd.concat([df for df in table_frames.values() if len(df) > 0], ignore_index=True) if table_frames else pd.DataFrame([])
    combined_path = family_out_dir / f'{family_id}_combined.csv'
    save_table(combined, combined_path)

    status_df = pd.DataFrame(status_rows)
    save_table(status_df, family_out_dir / f'{family_id}_status.csv')

    cache_hits_after = FEATURE_CACHE_STATS['hits']
    cache_misses_after = FEATURE_CACHE_STATS['misses']
    family_cache_hits = int(cache_hits_after - cache_hits_before)
    family_cache_misses = int(cache_misses_after - cache_misses_before)
    if _FEATURE_CACHE.enabled(family_id):
        total_requests = family_cache_hits + family_cache_misses
        hit_rate = (100.0 * family_cache_hits / total_requests) if total_requests > 0 else 0.0
        print(
            f"Feature cache stats [{family_id}] -> hits={family_cache_hits:,}, misses={family_cache_misses:,}, "
            f"hit_rate={hit_rate:.1f}%, entries={len(FEATURE_CACHE):,}"
        )

    return table_frames, combined, status_df


def run_all_families(default_threshold=0.5):
    all_family_combined = []
    all_status = []

    for family_id, family_cfg in FAMILIES.items():
        print('\n' + '=' * 100)
        print(f"FAMILY: {family_id} | {family_cfg['label']}")
        print('=' * 100)
        try:
            _, family_combined_df, family_status_df = evaluate_family(
                family_id=family_id,
                family_cfg=family_cfg,
                default_threshold=default_threshold,
            )
            if len(family_combined_df) > 0:
                all_family_combined.append(family_combined_df)
            all_status.append(family_status_df)
        except Exception as exc:
            all_status.append(pd.DataFrame([{
                'family_id': family_id,
                'table_name': '__family_failure__',
                'skipped_existing': False,
                'row_count': 0,
                'csv_path': '',
                'error': str(exc),
            }]))
            print(f'ERROR in family {family_id}: {exc}')

    global_combined_df = pd.concat(all_family_combined, ignore_index=True) if all_family_combined else pd.DataFrame([])
    global_status_df = pd.concat(all_status, ignore_index=True) if all_status else pd.DataFrame([])

    save_table(global_combined_df, EVAL_DIR / 'all_families_combined.csv')
    save_table(global_status_df, EVAL_DIR / 'all_families_status.csv')

    print('\n' + '=' * 100)
    print('GLOBAL SUMMARY')
    print('=' * 100)
    print(f'Combined rows: {len(global_combined_df):,}')
    print(f'Status rows:   {len(global_status_df):,}')
    print(
        f"Feature cache totals -> hits={FEATURE_CACHE_STATS['hits']:,}, "
        f"misses={FEATURE_CACHE_STATS['misses']:,}, entries={len(FEATURE_CACHE):,}"
    )

    return global_combined_df, global_status_df


# Run this cell when you are ready for full evaluation.
global_combined_results_df, global_status_df = run_all_families(default_threshold=0.5)

#print('Pipeline implementation complete. Uncomment the last line to execute full evaluation.')


FAMILY: baseline | SGD Baseline
SKIP existing table: baseline_own_year.csv
SKIP existing table: baseline_prior_years.csv
SKIP existing table: baseline_next_year.csv
SKIP existing table: baseline_cumulative.csv
SKIP existing table: baseline_final_model_each_year.csv
ERROR in family baseline: No columns to parse from file

FAMILY: huber | SGD Huber


C:\Users\bartu\AppData\Local\Temp\ipykernel_30208\2260530053.py:77: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)


ERROR in family huber: X has 22 features, but StandardScaler is expecting 14 features as input.

FAMILY: lagged_features | SGD Lagged Features
SKIP existing table: lagged_features_own_year.csv
SKIP existing table: lagged_features_prior_years.csv
SKIP existing table: lagged_features_next_year.csv
SKIP existing table: lagged_features_cumulative.csv
SKIP existing table: lagged_features_final_model_each_year.csv
ERROR in family lagged_features: No columns to parse from file

FAMILY: lagged_features_incremental_scaler | SGD Lagged Features (Incremental Scaler)
SKIP existing table: lagged_features_incremental_scaler_own_year.csv
SKIP existing table: lagged_features_incremental_scaler_prior_years.csv
SKIP existing table: lagged_features_incremental_scaler_next_year.csv
SKIP existing table: lagged_features_incremental_scaler_cumulative.csv
SKIP existing table: lagged_features_incremental_scaler_final_model_each_year.csv
ERROR in family lagged_features_incremental_scaler: No columns to parse fr

C:\Users\bartu\AppData\Local\Temp\ipykernel_30208\2260530053.py:77: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)


In [ ]:
# Section: Table-structured metric matrices (years as columns)
# F1 and PR AUC are displayed as separate tables.

summary_root = EVAL_DIR / 'summary_matrices'
summary_root.mkdir(parents=True, exist_ok=True)

# Remove legacy single prior-years exports to enforce per-model-year output format.
legacy_prior_paths = [
    summary_root / 'metric_matrix_prior_years_f1.csv',
    summary_root / 'metric_matrix_prior_years_pr_auc.csv',
]
for legacy_path in legacy_prior_paths:
    if legacy_path.exists():
        legacy_path.unlink()
        print(f'Removed legacy export: {legacy_path.name}')

required_cols = ['family_id', 'table_type', 'eval_year', 'f1_score', 'pr_auc']
all_frames = []
skipped_families = []

for family_id, family_cfg in FAMILIES.items():
    family_dir = EVAL_DIR / family_id
    combined_csv = family_dir / f'{family_id}_combined.csv'

    df = None
    reason = None

    if combined_csv.exists():
        try:
            df = pd.read_csv(combined_csv)
        except Exception as exc:
            reason = f'failed reading combined csv ({exc})'
    else:
        fallback_files = sorted(family_dir.glob(f'{family_id}_*.csv')) if family_dir.exists() else []
        fallback_files = [p for p in fallback_files if not p.name.endswith('_status.csv') and not p.name.endswith('_combined.csv')]
        if fallback_files:
            frames = []
            for file_path in fallback_files:
                try:
                    frames.append(pd.read_csv(file_path))
                except Exception:
                    pass
            if frames:
                df = pd.concat(frames, ignore_index=True)
            else:
                reason = 'fallback table csv files could not be read'
        else:
            reason = 'combined csv not found'

    if df is None or df.empty:
        skipped_families.append((family_id, reason or 'empty dataframe'))
        continue

    missing_required = [c for c in required_cols if c not in df.columns]
    if missing_required:
        skipped_families.append((family_id, f'missing columns: {missing_required}'))
        continue

    # model_year is required for per-model-year prior tables; keep NaN for table types where unavailable.
    if 'model_year' not in df.columns:
        df['model_year'] = np.nan

    df = df[required_cols + ['model_year']].copy()
    df['family_id'] = df['family_id'].fillna(family_id).astype(str)
    df['table_type'] = df['table_type'].astype(str)
    df['eval_year'] = pd.to_numeric(df['eval_year'], errors='coerce')
    df['model_year'] = pd.to_numeric(df['model_year'], errors='coerce')
    df['f1_score'] = pd.to_numeric(df['f1_score'], errors='coerce')
    df['pr_auc'] = pd.to_numeric(df['pr_auc'], errors='coerce')
    df = df.dropna(subset=['eval_year'])
    all_frames.append(df)

if not all_frames:
    print('No usable evaluation outputs found for matrix generation.')
else:
    results_df = pd.concat(all_frames, ignore_index=True)

    # Keep standard matrix exports for non-prior table types.
    table_types = [
        'own_year',
        'next_year',
        'cumulative',
        'final_model_each_year',
    ]

    metric_specs = [
        ('f1_score', 'f1'),
        ('pr_auc', 'pr_auc'),
    ]

    # Display/export labels for RR replay families (including PR variants); base replay rows remain unchanged.
    # Every family now carries its own display label (src/eval/families.py);
    # this used to be a second, separately-maintained, only-partially-covering
    # rename dict here. Derived, not hand-written, so a family added without a
    # short_label is a build-time omission instead of a raw id in a report.
    row_label_renames = {fid: cfg['short_label'] for fid, cfg in FAMILIES.items()}

    summary_outputs = {}

    for table_type in table_types:
        table_slice = results_df[results_df['table_type'] == table_type].copy()
        if table_slice.empty:
            print(f'[{table_type}] No rows available.')
            continue

        print('\n' + '=' * 100)
        print(f'TABLE: {table_type}')
        print('Rows are family_id. Columns are evaluation years.')
        print('Duplicate family-year rows are reduced using mean across contributing rows.')
        print('=' * 100)

        for metric_col, metric_label in metric_specs:
            metric_df = table_slice[['family_id', 'eval_year', metric_col]].copy()
            metric_df = metric_df.dropna(subset=[metric_col])

            if metric_df.empty:
                print(f'[{table_type}] {metric_label}: no non-null values available.')
                continue

            matrix_df = (
                metric_df
                .groupby(['family_id', 'eval_year'], as_index=False)[metric_col]
                .mean()
                .pivot(index='family_id', columns='eval_year', values=metric_col)
                .sort_index(axis=0)
                .sort_index(axis=1)
            )

            matrix_df.columns = [str(int(c)) if pd.notna(c) else str(c) for c in matrix_df.columns]
            matrix_df = matrix_df.reset_index().rename(columns={'family_id': 'row'})
            matrix_df['row'] = matrix_df['row'].replace(row_label_renames)

            summary_outputs[(table_type, metric_label)] = matrix_df

            export_path = summary_root / f'metric_matrix_{table_type}_{metric_label}.csv'
            matrix_df.to_csv(export_path, index=False)

            print(f'\n{metric_label.upper()} matrix')
            display(matrix_df)
            print(f'Saved: {export_path}')

    # Dedicated end section: one prior-years table per model year (columns grow with model year).
    prior_slice = results_df[results_df['table_type'] == 'prior_years'].copy()

    print('\n' + '=' * 100)
    print('TABLE SECTION: prior_years (per model year)')
    print('Each table is one model year. Rows are family_id. Columns are eval_year <= model_year.')
    print('=' * 100)

    if prior_slice.empty:
        print('[prior_years] No rows available.')
    elif prior_slice['model_year'].dropna().empty:
        print('[prior_years] Missing model_year values; cannot build per-model-year tables.')
    else:
        model_years = sorted(prior_slice['model_year'].dropna().astype(int).unique())

        for model_year in model_years:
            model_slice = prior_slice[prior_slice['model_year'].astype('Int64') == model_year].copy()
            if model_slice.empty:
                continue

            print('\n' + '-' * 100)
            print(f'PRIOR MODEL YEAR: {model_year}')
            print('-' * 100)

            for metric_col, metric_label in metric_specs:
                metric_df = model_slice[['family_id', 'eval_year', metric_col]].copy()
                metric_df = metric_df.dropna(subset=[metric_col])

                if metric_df.empty:
                    print(f'[prior_years_model_{model_year}] {metric_label}: no non-null values available.')
                    continue

                matrix_df = (
                    metric_df
                    .groupby(['family_id', 'eval_year'], as_index=False)[metric_col]
                    .mean()
                    .pivot(index='family_id', columns='eval_year', values=metric_col)
                    .sort_index(axis=0)
                    .sort_index(axis=1)
                )

                matrix_df.columns = [str(int(c)) if pd.notna(c) else str(c) for c in matrix_df.columns]
                matrix_df = matrix_df.reset_index().rename(columns={'family_id': 'row'})
                matrix_df['row'] = matrix_df['row'].replace(row_label_renames)

                table_key = f'prior_years_model_{model_year}'
                summary_outputs[(table_key, metric_label)] = matrix_df

                export_path = summary_root / f'metric_matrix_{table_key}_{metric_label}.csv'
                matrix_df.to_csv(export_path, index=False)

                print(f'\n{metric_label.upper()} matrix')
                display(matrix_df)
                print(f'Saved: {export_path}')

    if skipped_families:
        print('\nSkipped families (missing/invalid outputs):')
        for family_name, reason in skipped_families:
            print(f' - {family_name}: {reason}')

    overall_summary = pd.DataFrame([
        {
            'table_type': table_type,
            'metric': metric_label,
            'row_count': len(df_out),
        }
        for (table_type, metric_label), df_out in summary_outputs.items()
    ])

    overall_export_path = summary_root / 'metric_matrix_overview.csv'
    overall_summary.to_csv(overall_export_path, index=False)

    print('\nSummary matrix exports complete.')
    print(f'Output folder: {summary_root}')
    display(overall_summary.sort_values(['table_type', 'metric']).reset_index(drop=True))


TABLE: own_year
Rows are family_id. Columns are evaluation years.
Duplicate family-year rows are reduced using mean across contributing rows.

F1 matrix


,row,2017,2018,2019,2020,2021,2022
0,baseline,0.181211,0.394452,0.218509,0.256754,0.230862,0.258414
1,lagged_features,0.183724,0.394536,0.393827,0.419739,0.408259,0.413103
2,lagged_features_incremental_scaler,0.199615,0.395409,0.394097,0.412271,0.401910,0.411766
3,mlp_prevyears_monthly_features_incremental_scaler,0.263363,0.450257,0.468561,0.486295,0.470570,0.473149
4,mlp_experience_replay_RR_0.2,0.253968,0.449656,0.468287,0.493626,0.475431,0.493281
5,mlp_experience_replay_RR_0.2_PR_0.15,0.254553,0.443568,0.476202,0.484972,0.476525,0.475977
6,mlp_experience_replay_RR_0.3,0.257549,0.452124,0.470972,0.492685,0.475714,0.472905
7,mlp_experience_replay_RR_0.3_PR_0.10,0.258759,0.443730,0.461814,0.491207,0.476782,0.478229
8,mlp_experience_replay_RR_0.3_PR_0.15,0.267114,0.445327,0.465927,0.484842,0.477080,0.480458
9,mlp_experience_replay_RR_0.4,0.261505,0.450813,0.462095,0.484699,0.470926,0.469834


Saved: eval_outputs\unified_eval\summary_matrices\metric_matrix_own_year_f1.csv

PR_AUC matrix


,row,2017,2018,2019,2020,2021,2022
0,baseline,0.102965,0.379073,0.160839,0.193445,0.163037,0.195861
1,lagged_features,0.109286,0.378105,0.343162,0.385600,0.395786,0.386227
2,lagged_features_incremental_scaler,0.121508,0.375879,0.341089,0.376934,0.386522,0.386627
3,mlp_prevyears_monthly_features_incremental_scaler,0.203730,0.444538,0.425024,0.454080,0.440404,0.454931
4,mlp_experience_replay_RR_0.2,0.194117,0.442351,0.431550,0.467488,0.438441,0.472962
5,mlp_experience_replay_RR_0.2_PR_0.15,0.193359,0.429477,0.438943,0.454944,0.440482,0.455022
6,mlp_experience_replay_RR_0.3,0.184205,0.441084,0.431101,0.451725,0.439322,0.452903
7,mlp_experience_replay_RR_0.3_PR_0.10,0.185111,0.433883,0.422023,0.455221,0.450059,0.457294
8,mlp_experience_replay_RR_0.3_PR_0.15,0.201416,0.437488,0.421863,0.452702,0.449200,0.453089
9,mlp_experience_replay_RR_0.4,0.195044,0.443769,0.427322,0.453357,0.437619,0.458092


Saved: eval_outputs\unified_eval\summary_matrices\metric_matrix_own_year_pr_auc.csv

TABLE: next_year
Rows are family_id. Columns are evaluation years.
Duplicate family-year rows are reduced using mean across contributing rows.

F1 matrix


...

,row,2017,2018,2019,2020,2021,2022
0,baseline,0.176663,0.236311,0.204680,0.221384,0.207831,0.258414
1,lagged_features,0.158352,0.363753,0.389842,0.389340,0.399286,0.413103
2,lagged_features_incremental_scaler,0.157600,0.359747,0.390527,0.402285,0.386724,0.411766
3,mlp_prevyears_monthly_features_incremental_scaler,0.150726,0.181082,0.448506,0.489963,0.446812,0.473149
4,mlp_experience_replay_RR_0.2,0.224603,0.433717,0.453624,0.489852,0.462159,0.493281
5,mlp_experience_replay_RR_0.2_PR_0.15,0.231486,0.437947,0.460876,0.501897,0.473289,0.475977
6,mlp_experience_replay_RR_0.3,0.241057,0.442076,0.449680,0.489239,0.450895,0.472905
7,mlp_experience_replay_RR_0.3_PR_0.10,0.240485,0.439333,0.462606,0.496699,0.459576,0.478229
8,mlp_experience_replay_RR_0.3_PR_0.15,0.227659,0.428677,0.459732,0.499744,0.477289,0.480458
9,mlp_experience_replay_RR_0.4,0.248817,0.435882,0.447548,0.490654,0.465240,0.469834


Saved: eval_outputs\unified_eval\summary_matrices\metric_matrix_prior_years_model_2022_f1.csv

PR_AUC matrix


,row,2017,2018,2019,2020,2021,2022
0,baseline,0.114295,0.190124,0.148612,0.163078,0.147342,0.195861
1,lagged_features,0.086834,0.336067,0.337247,0.353612,0.372492,0.386227
2,lagged_features_incremental_scaler,0.085307,0.332058,0.339775,0.368690,0.368186,0.386627
3,mlp_prevyears_monthly_features_incremental_scaler,0.078806,0.108023,0.388679,0.458508,0.407686,0.454931
4,mlp_experience_replay_RR_0.2,0.155438,0.422557,0.409945,0.458534,0.430059,0.472962
5,mlp_experience_replay_RR_0.2_PR_0.15,0.154271,0.427455,0.417909,0.470253,0.438639,0.455022
6,mlp_experience_replay_RR_0.3,0.173606,0.441162,0.403056,0.459662,0.423617,0.452903
7,mlp_experience_replay_RR_0.3_PR_0.10,0.162300,0.432119,0.420626,0.467609,0.425877,0.457294
8,mlp_experience_replay_RR_0.3_PR_0.15,0.153024,0.417023,0.420012,0.467665,0.442473,0.453089
9,mlp_experience_replay_RR_0.4,0.183954,0.426867,0.406890,0.463793,0.430568,0.458092


Saved: eval_outputs\unified_eval\summary_matrices\metric_matrix_prior_years_model_2022_pr_auc.csv

Skipped families (missing/invalid outputs):
 - huber: combined csv not found
 - mlp_prevyears_monthly_features_incremental_scaler_experience_replay: combined csv not found
 - mlp_prevyears_monthly_features_incremental_scaler_experience_replay_RR_0.2_PR_0.10: combined csv not found
 - mlp_prevyears_monthly_features_incremental_scaler_experience_replay_RR_0.5_PR_0.10: combined csv not found
 - mlp_prevyears_monthly_features_incremental_scaler_experience_replay_uncertainity_prioritization_RR_0.2: combined csv not found
 - mlp_prevyears_monthly_features_incremental_scaler_experience_replay_uncertainity_prioritization_RR_0.5: combined csv not found

Summary matrix exports complete.
Output folder: eval_outputs\unified_eval\summary_matrices


,table_type,metric,row_count
0,cumulative,f1,35
1,cumulative,pr_auc,35
2,final_model_each_year,f1,27
3,final_model_each_year,pr_auc,27
4,next_year,f1,35
5,next_year,pr_auc,35
6,own_year,f1,35
7,own_year,pr_auc,35
8,prior_years_model_2017,f1,35
9,prior_years_model_2017,pr_auc,35
